# Setup

In [ ]:
# Fabric notebook parameters. The pipeline's ForEach passes one source per
# iteration, so a failure is isolated to that source; None means "every enabled
# source", which is what a manual run wants.
#
# This cell must stay tagged `parameters` — Fabric injects the pipeline's values
# in a new cell directly below it, so anything defined here is a default, not a
# constant.
source_name = None

In [ ]:
# Environment bootstrap. A Fabric notebook has neither the repo root on
# sys.path nor as its working directory, so relative paths like
# "config/config.yaml" cannot resolve there. Detecting the OneLake mount keeps
# one notebook working in both places instead of maintaining two copies.
import os
import sys

CODE_ROOT = "/lakehouse/default/Files/code"
IN_FABRIC = os.path.isdir(CODE_ROOT)

if IN_FABRIC and CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

CONFIG_DIR = f"{CODE_ROOT}/config" if IN_FABRIC else "config"

# In Fabric, secrets come from the workspace rather than a gitignored .env.
# Set them here for a trial run; Chapter 8 replaces this with Key Vault.
if IN_FABRIC:
    os.environ["ADZUNA_APP_ID"]  = "2961c1f9>"
    os.environ["ADZUNA_APP_KEY"] = "79aad18b4b9315f10de75c2b1b34ac30"
    os.environ["JOOBLE_API_KEY"] = "806fb85d-5f54-4493-9272-c9aef30dd68c"
    os.environ.setdefault("APP_ENV", "fabric")

print("Running in Fabric" if IN_FABRIC else "Running locally")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from transformation.bronze_writer import run_bronze_ingestion
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger

setup_logging(config_path=f"{CONFIG_DIR}/logging_config.yaml")
logger = get_logger(__name__)

# Fabric provides a Delta-enabled session already; getOrCreate() returns it.
spark = SparkSession.builder.appName("bronze_ingestion").getOrCreate()

app_config = load_config(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
)

# Paths

In [ ]:
# The same OneLake location needs two different path forms, depending on which
# library does the reading. Plain Python IO (run_ingestion's json.dump,
# bronze_writer's Path.glob) needs the local mount; Spark takes the path
# relative to the notebook's default lakehouse. Mixing them up produces an
# empty DataFrame rather than an error, so it is worth being explicit.
BRONZE_JSON_DIR = "/lakehouse/default/Files/bronze" if IN_FABRIC else "data/bronze"
BRONZE_TABLE_PATH = "Tables/bronze_job_postings" if IN_FABRIC else "data/delta/bronze_job_postings"

# Run ingestion (Chapter 2's connectors), then write Bronze

`run()` takes every path as an argument rather than loading config off a
hardcoded relative path — that is what lets this same call work from a
Fabric notebook, a local CLI run, and CI without any of them needing to
chdir or edit files on disk.

In [ ]:
from ingestion.run_ingestion import run as run_connectors

exit_code = run_connectors(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
    logging_config_path=f"{CONFIG_DIR}/logging_config.yaml",
    bronze_path=BRONZE_JSON_DIR,
    source_name=source_name,
)

# 0 = every source succeeded; 1 = at least one source failed (see logs above).
assert exit_code == 0, f"Ingestion reported failures, exit_code={exit_code}"


# Read the connector JSON output and write the Bronze Delta table

In [ ]:
bronze_df = run_bronze_ingestion(
    spark,
    json_dir=BRONZE_JSON_DIR,
    table_path=BRONZE_TABLE_PATH,
)
bronze_df.show(5, truncate=50)

# Sanity checks before trusting this run

A row count of 0 here almost always means `BRONZE_JSON_DIR` is wrong, not
that the APIs returned nothing: `Path.glob` on a missing directory yields
nothing silently instead of raising.

In [ ]:
print("Row count:", bronze_df.count())
print("Schema:")
bronze_df.printSchema()
print("Records per source:")
bronze_df.groupBy("source").count().show()